# 🧬 AMR-Predictor-ML: Predicting Meropenem Resistance in *Klebsiella pneumoniae*
## Phase 3: Machine Learning Baseline Model Training

### 📋 Phase Overview
Now that we have a clean, strict ACGT 6-mer feature matrix ($2836 \times 4101$), we are ready to build our predictive models. In this phase, we will split our dataset into training and testing sets, handle feature scaling, and train a baseline **Random Forest Classifier** which is highly effective for high-dimensional genomic tabular data.

### 🎯 Notebook Objectives
1. Split the golden dataset into Stratified Train/Test sets (80/20) to preserve class balance.
2. Separate metadata/identifiers from mathematical features ($X$) and target labels ($y$).
3. Train a baseline Random Forest model.
4. Evaluate performance using professional classification metrics (Accuracy, Precision, Recall, F1-Score, and ROC-AUC).

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load the golden dataset we built in Phase 2
dataset_path = "../data/processed/kmers_6_dataset.csv"
print("⏳ Loading Golden Dataset...")
df = pd.read_csv(dataset_path)

# 2. Separate Target, Metadata, and Features
# We define the columns that are NOT mathematical features for the ML model
metadata_cols = ['Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype', 'AMR_Label']

y = df['AMR_Label'].values  # Target Vector (0 for Susceptible, 1 for Resistant)
X = df.drop(columns=metadata_cols).values  # Feature Matrix (Only the 4,096 K-mer counts)

print(f"✅ Data Separated Successfully!")
print(f"📊 Feature Matrix Shape (X): {X.shape}")
print(f"🎯 Target Vector Shape (y) : {y.shape}")

# 3. Stratified Train/Test Split (80% Train, 20% Test)
# - test_size=0.20 sets exactly 20% of data for evaluation.
# - random_state=42 is a seed that ensures we get the exact same split if we re-run the code.
# - stratify=y guarantees both sets maintain the exact same 62% vs 38% class ratio.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("\n⚔️ --- Train/Test Split Summary ---")
print(f"🏋️ Training Set (X_train): {X_train.shape[0]} samples")
print(f"🧪 Testing Set  (X_test) : {X_test.shape[0]} samples")
print(f"📈 Training Resistance Ratio: {np.mean(y_train)*100:.2f}%")
print(f"📉 Testing Resistance Ratio : {np.mean(y_test)*100:.2f}%")

⏳ Loading Golden Dataset...
✅ Data Separated Successfully!
📊 Feature Matrix Shape (X): (2836, 4096)
🎯 Target Vector Shape (y) : (2836,)

⚔️ --- Train/Test Split Summary ---
🏋️ Training Set (X_train): 2268 samples
🧪 Testing Set  (X_test) : 568 samples
📈 Training Resistance Ratio: 37.79%
📉 Testing Resistance Ratio : 37.85%


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# 1. Initialize the Baseline Random Forest Classifier
# - n_estimators=100: We will build 100 decision trees in our forest.
# - random_state=42: To ensure identical results every time we train.
# - n_jobs=-1: Use ALL available CPU cores of the Azure VM to speed up training.
print("🏋️ Training the Baseline Random Forest Model on genomic features...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Fit the model on training data
rf_model.fit(X_train, y_train)
print("✅ Training complete!")

# 2. Make Predictions on the Unseen Test Set
print("🧪 Evaluating model performance on the unseen Testing Set...")
y_pred = rf_model.predict(X_test)         # Binary outputs (0 or 1)
y_prob = rf_model.predict_proba(X_test)[:, 1] # Probability outputs for AUC calculation

# 3. Print Professional Performance Summary Metrics
print("\n📊 === Machine Learning Evaluation Summary ===")
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"🎯 Global Accuracy : {accuracy*100:.2f}%")
print(f"📈 ROC-AUC Score   : {roc_auc*100:.2f}%")

print("\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Susceptible (0)", "Resistant (1)"]))

🏋️ Training the Baseline Random Forest Model on genomic features...
✅ Training complete!
🧪 Evaluating model performance on the unseen Testing Set...

📊 === Machine Learning Evaluation Summary ===
🎯 Global Accuracy : 66.90%
📈 ROC-AUC Score   : 68.00%

📋 Detailed Classification Report:
                 precision    recall  f1-score   support

Susceptible (0)       0.69      0.86      0.76       353
  Resistant (1)       0.61      0.35      0.45       215

       accuracy                           0.67       568
      macro avg       0.65      0.61      0.61       568
   weighted avg       0.66      0.67      0.64       568



In [4]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# 1. Initialize the XGBoost Classifier
# - n_estimators=200: Build 200 sequential trees.
# - learning_rate=0.1: How aggressive the model corrects mistakes.
# - max_depth=6: Maximum depth of a tree (prevents overfitting).
print("🚀 Training the XGBoost Model on genomic features...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200, 
    learning_rate=0.1, 
    max_depth=6, 
    random_state=42, 
    n_jobs=-1,
    eval_metric='logloss'
)

# 2. Fit the model on training data
xgb_model.fit(X_train, y_train)
print("✅ XGBoost Training complete!")

# 3. Make Predictions on the Unseen Test Set
print("🧪 Evaluating XGBoost performance on the Testing Set...")
y_pred_xgb = xgb_model.predict(X_test)         
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1] 

# 4. Print Professional Performance Summary Metrics
print("\n📊 === XGBoost Evaluation Summary ===")
acc_xgb = accuracy_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

print(f"🎯 Global Accuracy : {acc_xgb*100:.2f}%")
print(f"📈 ROC-AUC Score   : {auc_xgb*100:.2f}%")

print("\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=["Susceptible (0)", "Resistant (1)"]))

🚀 Training the XGBoost Model on genomic features...
✅ XGBoost Training complete!
🧪 Evaluating XGBoost performance on the Testing Set...

📊 === XGBoost Evaluation Summary ===
🎯 Global Accuracy : 68.66%
📈 ROC-AUC Score   : 74.49%

📋 Detailed Classification Report:
                 precision    recall  f1-score   support

Susceptible (0)       0.72      0.81      0.76       353
  Resistant (1)       0.61      0.49      0.54       215

       accuracy                           0.69       568
      macro avg       0.66      0.65      0.65       568
   weighted avg       0.68      0.69      0.68       568

